# Stage Movement and Sample Mapping

Move the stage, acquire overview images, and return to the starting position while mapping the sample.


### Run the servers

Make sure you are on the VPN and the AutoScript server is running. Then start the asyncroscopy Tango servers from the repository root:

```bash
uv run scripts/3_run_servers.py
```


### Imports


In [1]:
import os
import json
import time

import tango
import numpy as np
import matplotlib.pyplot as plt
from tiled.client import from_uri

%matplotlib ipympl


### Ping servers


In [ ]:
DB_HOST = "10.46.217.241"
DB_PORT = 9094

os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"

server_names = ['stage', 'scan', 'eds', 'camera', 'data', 'microscope']

for name in server_names:
    device_name = f"asyncroscopy/{name}/default"
    proxy = tango.DeviceProxy(device_name)
    proxy.ping()
    print(device_name, proxy.state())


### Connect to devices


In [ ]:
scan = tango.DeviceProxy("asyncroscopy/scan/default")
microscope = tango.DeviceProxy("asyncroscopy/microscope/default")
data = tango.DeviceProxy("asyncroscopy/data/default")

# Backward-compatible aliases used by the workflow cells below.
mic_proxy = microscope
microscope_proxy = microscope

for proxy in (scan, microscope, data):
    proxy.set_timeout_millis(120_000)

print("scan      :", scan.state())
print("microscope:", microscope.state())
print("data      :", data.state())


### Start Tiled data server


In [2]:
TILED_HOST = "10.46.217.241"
TILED_PORT = 9091
save_path = "D:/microscopedata/tiled/ahoust17/2026_05_22_AtomFab/"

data.host = TILED_HOST
data.port = TILED_PORT
data.save_path = save_path

if str(data.tiled_server).lower() != "yes":
    print("Tiled server is not responding; starting it from the DATA device...")
    config = json.loads(data.start_tiled_server())
else:
    print("Tiled server is already running.")
    config = json.loads(data.get_config())

print(json.dumps(config, indent=2))


NameError: name 'data' is not defined

### Configure the scan


In [ ]:
scan.Activate(['haadf'])
scan.dwell_time   = 600e-6  # µs
scan.imsize  = 8

print('dwell_time  :', scan.dwell_time)
print('image size :', scan.imsize)


### Set the field of view


In [ ]:
# fov cutoff is 0.52e-9
fov = 0.52e-9
microscope_proxy.set_fov(fov)

### Acquire a reference image


In [ ]:
# get_image returns DevEncoded = (json_metadata_str, raw_bytes)
json_meta, raw_bytes = microscope_proxy.get_scanned_image()

metadata  = dict(json.loads(json_meta))
image = np.frombuffer(raw_bytes, dtype=metadata['dtype']).reshape(metadata['shape'])

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap='gray', interpolation='none')
ax.set_title(f"HAADF — dwell {metadata['dwell_time']*1e6:.1f} µs")
ax.axis('off')
plt.tight_layout()
plt.show()

### Read the current stage position


In [ ]:
# get the stage
starting_position = microscope_proxy.get_stage()
print('Stage position:', starting_position)


### Move the stage


In [ ]:
# move the stage
move_by = -100e-6  # 10 µm

new_position = starting_position + np.array([move_by, 0, 0, 1])

In [ ]:
microscope_proxy.move_stage(new_position)

### Return to the starting position


In [ ]:
microscope_proxy.move_stage(starting_position)